# exp_09 Feature Reduction

In [ ]:
# Install package from GitHub
!pip install git+https://github.com/SilasPignotti/urban-tree-transfer.git -q
# Optional dependencies
!pip install optuna pytorch-tabnet -q

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/dev/urban-tree-transfer')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'outputs/phase_3'

(OUTPUT_DIR / 'metadata').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'logs').mkdir(parents=True, exist_ok=True)

from urban_tree_transfer.config import load_experiment_config
from urban_tree_transfer.experiments import (
    ablation,
    data_loading,
    preprocessing,
    training,
    evaluation,
    visualization,
    models,
    transfer,
    hp_tuning,
)
from urban_tree_transfer.utils import (
    validate_setup_decisions,
    validate_algorithm_comparison,
    validate_hp_tuning_result,
    validate_evaluation_metrics,
    validate_finetuning_curve,
)

In [ ]:
# Load all prior decisions from exp_08, exp_08b, exp_08c
config = load_experiment_config()
setup_path = OUTPUT_DIR / 'metadata/setup_decisions.json'

if not setup_path.exists():
    raise FileNotFoundError(f"setup_decisions.json not found. Run exp_08, exp_08b, exp_08c first.")

setup = json.loads(setup_path.read_text())

required_keys = ['chm_strategy', 'proximity_strategy', 'outlier_strategy']
missing = [k for k in required_keys if k not in setup]
if missing:
    raise ValueError(f"Missing decisions: {missing}. Run exp_08, exp_08b, exp_08c in order.")

chm_strategy = setup['chm_strategy']['decision']
chm_features = setup['chm_strategy'].get('included_features', [])
proximity_strategy = setup['proximity_strategy']['decision']
outlier_strategy = setup['outlier_strategy']['decision']

print(f"Loaded decisions:")
print(f"  CHM: {chm_strategy}")
print(f"  Proximity: {proximity_strategy}")
print(f"  Outlier: {outlier_strategy}")

In [ ]:
# Apply all 3 decisions sequentially to get final dataset
# Load correct proximity variant
train_df, val_df, _ = data_loading.load_berlin_splits(
    DATA_DIR / 'phase_2_splits', variant=proximity_strategy
)

# Apply CHM strategy
train_df = ablation.apply_chm_strategy(train_df, chm_strategy)
val_df = ablation.apply_chm_strategy(val_df, chm_strategy)

# Apply outlier removal
train_df = ablation.apply_outlier_removal(train_df, outlier_strategy)
val_df = ablation.apply_outlier_removal(val_df, outlier_strategy)

# Get feature columns
feature_cols = data_loading.get_feature_columns(
    train_df,
    include_chm=bool(chm_features),
    chm_features=chm_features or None,
)

print(f"\nFinal dataset after all decisions:")
print(f"  Training samples: {len(train_df)}")
print(f"  Features: {len(feature_cols)}")

x_train = train_df[feature_cols]
x_val = val_df[feature_cols]

x_train_scaled, x_val_scaled, _, _ = preprocessing.scale_features(x_train, x_val=x_val)
y_train, label_to_idx, _ = preprocessing.encode_genus_labels(train_df['genus_latin'])

In [ ]:
# Compute feature importance on final dataset
model = models.create_model('random_forest')
model.fit(x_train_scaled, y_train)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

print(f"\nTop 10 most important features:")
print(importance_df.head(10).to_string(index=False))

# Visualize feature importance
fig_dir = OUTPUT_DIR / 'figures/exp_09_feature_reduction'
fig_dir.mkdir(parents=True, exist_ok=True)
visualization.plot_feature_importance(
    importance_df.head(30),
    output_path=fig_dir / 'feature_importance_top30.png',
)

In [ ]:
# Test candidate feature subset sizes
candidate_k = config['setup_ablation']['feature_reduction']['candidate_k']
cv = training.create_spatial_block_cv(train_df, n_splits=config['global']['cv_folds'])

results = []
for k in candidate_k:
    selected = importance_df.head(k)['feature'].tolist()
    x_train_k = train_df[selected]
    x_val_k = val_df[selected]
    x_train_k_scaled, x_val_k_scaled, _, _ = preprocessing.scale_features(x_train_k, x_val=x_val_k)

    rf = models.create_model('random_forest')
    metrics = training.train_with_cv(rf, x_train_k_scaled, y_train, train_df['block_id'].values, cv)

    results.append({
        'n_features': k,
        'val_f1_mean': metrics['val_f1_mean'],
        'val_f1_std': metrics['val_f1_std'],
        'train_val_gap': metrics['train_val_gap'],
    })

results_df = pd.DataFrame(results)
print(f"\nFeature subset evaluation:")
print(results_df.to_string(index=False))

In [ ]:
# Select optimal k using Pareto criterion
# Find smallest k where F1 >= best_f1 - max_drop
best_f1 = results_df['val_f1_mean'].max()
max_drop = 0.01  # Allow 1% F1 drop for simplicity

pareto_threshold = best_f1 - max_drop
optimal_row = results_df[results_df['val_f1_mean'] >= pareto_threshold].iloc[0]
selected_k = int(optimal_row['n_features'])
selected_f1 = optimal_row['val_f1_mean']

selected_features = importance_df.head(selected_k)['feature'].tolist()

print(f"\nOptimal feature set:")
print(f"  k = {selected_k} features")
print(f"  val_f1 = {selected_f1:.4f}")
print(f"  Pareto threshold (best - {max_drop}): {pareto_threshold:.4f}")

# Generate Pareto curve visualization
visualization.plot_pareto_curve(
    results_df,
    selected_k=selected_k,
    output_path=fig_dir / 'pareto_curve.png',
)

In [ ]:
# Update setup_decisions.json with feature_set and selected_features
setup['feature_set'] = {
    'n_features': selected_k,
    'reasoning': f"Pareto selection: smallest k with F1 >= {pareto_threshold:.4f}",
    'importance_ranking': importance_df.to_dict(orient='records'),
    'ablation_results': results_df.to_dict(orient='records'),
}

setup['selected_features'] = selected_features

setup_path.write_text(json.dumps(setup, indent=2))
print(f"\nCompleted setup_decisions.json with all 4 strategies + selected features")

# Validate against schema
validate_setup_decisions(setup_path)
print("Schema validation: PASSED")

print(f"\nSetup decisions summary:")
print(f"  CHM: {chm_strategy}")
print(f"  Proximity: {proximity_strategy}")
print(f"  Outlier: {outlier_strategy}")
print(f"  Features: {selected_k} selected")
print(f"\nReady for 03a_setup_fixation.ipynb")